# OpenAI Agents SDK + Databricks MCP Server

This notebook shows how to attach the custom MCP server (deployed as a Databricks App) to an agent built with the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/mcp/).

- **LLM**: a Databricks Model Serving endpoint, called through its OpenAI-compatible API.
- **Tools**: discovered automatically from the MCP server (`get_account_details`, `forecast_savings`, ...).
- **Auth**: Databricks OAuth via your local Databricks CLI profile (Databricks Apps only accept OAuth tokens).

Prerequisites: the app is deployed (`databricks bundle deploy && databricks bundle run mcp_server_template`) and you are logged in (`databricks auth login --profile <profile>`).

In [0]:
%pip install -q "openai-agents>=0.2.0" "databricks-sdk>=0.50.0"

## Configuration

In [0]:
PROFILE = "resa-prod"                      # Databricks CLI profile
APP_NAME = "mcp-server-template"          # Databricks App hosting the MCP server
MODEL_NAME = "gpt-5-6-sol" # any tool-calling serving endpoint
LOCAL_MCP = False                         # True -> use `make run` server on localhost:8000

In [0]:
from databricks.sdk import WorkspaceClient

if LOCAL_MCP:
    w = WorkspaceClient(profile=PROFILE)
    mcp_url, mcp_headers = "http://localhost:8000/mcp", {}
else:
    w = WorkspaceClient(
        host=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get(),  # current workspace
        client_id=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-id"),
        client_secret=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-secret"),
        auth_type="oauth-m2m",
        )
    mcp_url = w.apps.get(APP_NAME).url.rstrip("/") + "/mcp"
    mcp_headers = w.config.authenticate()  # {"Authorization": "Bearer <oauth-token>"}

print("Workspace:", w.config.host)
print("MCP URL:  ", mcp_url)

## LLM client

The Agents SDK defaults to OpenAI's Responses API. Databricks serving endpoints speak the Chat Completions API, so we wrap an `AsyncOpenAI` client pointing at the workspace in `OpenAIChatCompletionsModel`. Tracing to OpenAI is disabled since no OpenAI key is used.

In [0]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

set_tracing_disabled(True)

llm_client = AsyncOpenAI(
    base_url=f"{w.config.host}/serving-endpoints",
    api_key=w.config.authenticate()["Authorization"].removeprefix("Bearer "),
)
model = OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=llm_client)

## Connect to the MCP server and list its tools

In [0]:
from agents.mcp import MCPServerStreamableHttp
from databricks.sdk import WorkspaceClient

def make_mcp_server() -> MCPServerStreamableHttp:
    return MCPServerStreamableHttp(
        name="mcp-server-template",
        params={"url": mcp_url, "headers": mcp_headers, "timeout": 30},
        cache_tools_list=True,
    )

async with make_mcp_server() as server:
    for tool in await server.list_tools():
        print(f"- {tool.name}: {(tool.description or '').strip().splitlines()[0]}")

## Create the agent and run it

The MCP server is passed via `mcp_servers=[...]`; the SDK exposes its tools to the LLM and routes tool calls back to the server.

In [0]:
from agents import Agent, Runner, ModelSettings, set_default_openai_api
from openai.types.shared import Reasoning

set_default_openai_api("responses")


INSTRUCTIONS = """You are a helpful banking assistant.
Use the available tools to look up the user's account and to forecast savings.
Always state amounts with their currency and explain the assumptions you used."""

async def ask(question: str) -> str:
    async with make_mcp_server() as server:
        agent = Agent(
            name="Savings Assistant",
            instructions=INSTRUCTIONS,
            model=model,
            model_settings=ModelSettings(
                reasoning=Reasoning(effort="none"),
            ),
            mcp_servers=[server],
        )
        result = await Runner.run(agent, question)

    for item in result.new_items:
        if item.type == "tool_call_item":
            print(f"[tool call]   {item.raw_item.name}({item.raw_item.arguments})")
        elif item.type == "tool_call_output_item":
            print(f"[tool output] {str(item.output)[:200]}")
    return result.final_output

print(await ask("What is my current account balance?"))

In [0]:
print(await ask(
    "If I save 300 per month for 15 years with a 6% annual return, "
    "how much will I have? How much of that is interest?"
))

## Streaming events (optional)

`Runner.run_streamed` lets you observe tool calls and text as they happen.

In [0]:
from openai.types.responses import ResponseTextDeltaEvent

async with make_mcp_server() as server:
    agent = Agent(name="Savings Assistant", instructions=INSTRUCTIONS, model=model, mcp_servers=[server])
    stream = Runner.run_streamed(agent, "Compare saving 100 vs 500 per month for 10 years at 5%.")
    async for event in stream.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)
        elif event.type == "run_item_stream_event" and event.item.type == "tool_call_item":
            print(f"\n[tool call] {event.item.raw_item.name}({event.item.raw_item.arguments})")